# 19.10 如何把成品帶到較小的電腦？


一份大模型可用較少位元近似儲存，也可另外訓練較小學生。前者是量化，改變同一結構的數值；後者是蒸餾，學生結構先變小，再學教師訊號。兩條路各有用途，不必全部套在共同成品上才算完成。

刻度（scale）是相鄰整數格代表的浮點間隔；例如刻度0.5時，碼2還原成1.0。按刻度還原近似浮點值叫反量化，詳見[17.2](https://birdhackor.github.io/tiny-perceptron-vlm/17.2.html)。先看一層4×8權重，不牽涉整個助手：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

from tiny_perceptron.quantization import QuantizedLinear

torch.manual_seed(42)
layer = nn.Linear(8, 4)
compressed = QuantizedLinear(layer, bits=4)
x = torch.ones(1, 8)
with torch.no_grad():
    error = (layer(x) - compressed(x)).abs().max().item()
print("FP32權重加bias bytes", sum(p.numel() * p.element_size() for p in layer.parameters()))
print("打包權重、刻度與bias bytes", compressed.storage_bytes(), "最大輸出差", error)

FP32 的32個權重加4個 bias 共144 bytes。4-bit 打包權重16 bytes，本例另存4個FP32刻度，每個4 bytes，共16 bytes；加上4個 bias 的16 bytes，這個物件的張量儲存共48 bytes。原輸入是八個1；最大輸出差只針對這一筆，不是整份模型答錯率。這個可讀實作先把整數碼按刻度還原成近似浮點權重，再與輸入計算，所以檔案小不直接代表執行更快。逐輸出列的刻度見[17.5](https://birdhackor.github.io/tiny-perceptron-vlm/17.5.html)。

完整助手壓縮後要真的存檔、重載，再核對指令、指定框、語音、工具參數與 EOS。檔案大小、載入後張量與執行峯值各量各的；某題原本已錯，壓縮後仍錯而總分相同，不等於輸出完全一致。

若改成每個 token 都使用同一組前饋權重的 Dense 學生（見[15.1](https://birdhackor.github.io/tiny-perceptron-vlm/15.1.html)），則說明它與教師的架構、資料與訓練預算。教師固定，學生可同時學標準答案 CE 和教師分佈；舊配方用 `0.5 CE + 0.5 T² KL(p_T||q_T)`，T=2；p_T、q_T 分別是教師與學生在此溫度下的候選分佈。KL 量兩份分佈的差，見[18.8](https://birdhackor.github.io/tiny-perceptron-vlm/18.8.html)與[18.9](https://birdhackor.github.io/tiny-perceptron-vlm/18.9.html)。內部已乘 T²，不再乘一次。它也要與同尺寸只學標準答案的學生比，才能分清縮小和教師訊號的影響。新成品是否需要這條部署路線，由資源與能力取捨決定。

練習把 `bits=4` 改成 `bits=8`：先預測32個權重的打包儲存由16變32 bytes，加上原來的刻度和 bias，合計64 bytes，再執行核對。單筆最大輸出差是否變小，仍不能替整個助手的能力下結論。

<details>
<summary>補充：舊合成任務的量化與學生比較</summary>

訓練後量化（PTQ）是在既有權重上改用較少位元保存。舊推薦 joint 的量化版本已存檔、重新載入，再重跑同一份最後檢查；這與[19.9](https://birdhackor.github.io/tiny-perceptron-vlm/19.9.html)同一權重的快取一致對照是不同實驗。

總分相同仍可能改變輸出。例如「請算1加0」在完整工具迴圈讀回結果後，FP32 模型答 `111`，4-bit 版本答 `11`。兩個都錯，因此分數未變；不能據此說逐 token 相同或量化修好了原先的錯誤。儲存規格及逐題記錄見[部署實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/results/capstone_deployment.json)。

測試檔案包含容器與模型結構、來源等 metadata。公開版整理這些附加資訊後，檔案大小與指紋可能改變；下載應核對[19.11](https://birdhackor.github.io/tiny-perceptron-vlm/19.11.html)的正式清單。這些打包檔案載入後仍還原 FP32，也沒有專用低位元計算核心，不能把檔案縮小直接當成執行記憶體縮為八分之一。

學生比較使用[19.8](https://birdhackor.github.io/tiny-perceptron-vlm/19.8.html)的 DPO 比較分支作教師，與推薦 joint 權重有別。CE 學生只學標準答案；KD 學生同時學標準答案與教師分佈，使用主文的混合配方。兩支複製同一份隨機 Dense 起點，固定相同抽樣種子、批次規則與350次更新。這控制了配方，但報告沒有逐批保存 GPU 取樣清單，不能稱做過逐批實錄核對。KD 還需固定教師的前向計算，訓練成本不能只用學生參數數量估計。

「1+8等於多少？」能看出能力未必保留：CE 學生請求 `TOOL:calculator:1+8`，工具回9，最後答9；KD 學生卻請求 `TOOL:calculator:1+9`，工具因而回10，模型最後又答11。請求參數和讀回兩層都錯了，不能只看最後總分判斷是否保留教師能力。

這是固定預算、舊合成任務的小學生比較，不能推論所有 Dense 或蒸餾配方的結果。此處也沒有量學生的推論速度或生成記憶體峯值，參數較少不能直接換算成速度。完整條件與逐題記錄見[學生實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/results/capstone_student.json)。若要研究新壓縮配方，另切驗證／最後考卷；不要看這次最後題目後挑較好的一次重跑冒充原實驗。

</details>
